# LIGER default - nph_cerebellum

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'nph_cerebellum'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Sample'
LABEL_KEY = 'Celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'nph_cerebellum'
assert BATCH_KEY == 'Sample'
assert LABEL_KEY == 'Celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                         nph_cerebellum
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                               Sample
label_key                                             Celltype
metrics               [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=84619 n_batches=2 n_labels=8
CPCR preflight: eligible_labels=8


In [4]:
LIGER_DEFAULT_ARGS = {
    "k": 20,
    "value_lambda": 5.0,
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_default(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_DEFAULT_ARGS["k"],
        lam=LIGER_DEFAULT_ARGS["value_lambda"],
        max_iterations=LIGER_DEFAULT_ARGS["max_iters"],
        mode=LIGER_DEFAULT_ARGS["mode"],
        layer=LIGER_DEFAULT_ARGS["layer"],
        seed=seed,
    )


pd.Series(LIGER_DEFAULT_ARGS, name="default_setting")


k                   20
value_lambda       5.0
max_iters           30
mode              core
layer           counts
Name: default_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 1023 genes not expressing in FM3.


Removing 246 genes not expressing in MM1.


[LIGER PARAMS] function=optimize_ALS k=20 value_lambda=5.0 max_iters=30 rand_seed=1


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7327459620713402


optimised clustering against liger_metric_label
optimal cluster resolution: 0.2
optimal score: 0.7871563666119501
BRAS...


seed           10.000000
BAMI            0.787156
CASW            0.785641
SCGRAPHM        0.383329
BRAS            0.875592
KBET            0.697236
CPCR            0.217986
bio_score       0.652042
batch_score     0.596938
joint_score     0.630001
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.792240,0.791940,0.385928,0.871827,0.662997,0.149700,0.656703,0.561508,0.618625
1,2,0.775101,0.763847,0.462604,0.844292,0.689364,0.070653,0.667184,0.534769,0.614218
2,3,0.783473,0.831708,0.424301,0.862478,0.619516,0.475901,0.679827,0.652631,0.668949
3,4,0.818423,0.810065,0.369165,0.846228,0.685910,0.390821,0.665884,0.640986,0.655925
4,5,0.766974,0.800844,0.356768,0.792666,0.686762,0.532776,0.641529,0.670735,0.653211
5,6,0.780726,0.811975,0.412460,0.800622,0.721172,0.397298,0.668387,0.639697,0.656911
6,7,0.813867,0.795767,0.370584,0.835463,0.661107,0.276032,0.660073,0.590867,0.632390
7,8,0.768739,0.815509,0.475992,0.804225,0.730703,0.070316,0.686747,0.535081,0.626080
8,9,0.790063,0.806228,0.387219,0.853150,0.703475,0.249315,0.661170,0.601980,0.637494
9,10,0.787156,0.785641,0.383329,0.875592,0.697236,0.217986,0.652042,0.596938,0.630001


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.787676,0.016338,0.766974,0.818423
CASW,0.801352,0.017629,0.763847,0.831708
SCGRAPHM,0.402835,0.038319,0.356768,0.475992
BRAS,0.838654,0.028440,0.792666,0.875592
KBET,0.685824,0.030410,0.619516,0.730703
CPCR,0.283080,0.154205,0.070316,0.532776
bio_score,0.663955,0.012351,0.641529,0.686747
batch_score,0.602519,0.045841,0.534769,0.670735
joint_score,0.639380,0.017401,0.614218,0.668949
